In [1]:
import pandas as pd
import numpy as np
from scipy.optimize import minimize
import multiprocessing as mp

In [2]:
# Function to load and preprocess data
def load_and_preprocess_data(file_path):
    # Load data
    df = pd.read_csv(file_path)

    # Drop unnecessary columns
    # Assuming the first two columns are not required (based on the dataset structure observed)
    df.drop(df.columns[[0, 1]], axis=1, inplace=True)

    # Handle missing values
    # You can choose different strategies like filling with mean/median, forward fill, or dropping them
    # Here, let's forward fill the missing values, then backfill, and eventually drop any remaining NaNs
    df.fillna(method='ffill', inplace=True)
    df.fillna(method='bfill', inplace=True)
    df.dropna(axis=1, inplace=True)

    return df

In [3]:
def calculate_profit_loss(prices, k):
    # Assuming the DataFrame 'prices' has two columns, one for each asset
    # The first column is the asset we go long on, and the second is the one we go short on

    # Calculate the number of shares bought for the first asset and sold for the second asset
    # At the start of the investment period
    shares_asset1 = k / prices.iloc[0, 0]  # Number of shares for the first asset
    shares_asset2 = k / prices.iloc[0, 1]  # Number of shares for the second asset

    # Calculate the profit or loss for each asset at the end of the investment period
    profit_loss_asset1 = (prices.iloc[-1, 0] - prices.iloc[0, 0]) * shares_asset1
    profit_loss_asset2 = (prices.iloc[0, 1] - prices.iloc[-1, 1]) * shares_asset2  # Short position

    # Total profit or loss
    profit_loss = profit_loss_asset1 + profit_loss_asset2

    return profit_loss

In [13]:
def optimize_pairs(df, theta, k, max_iter=1000, learning_rate=0.01):
    # Define the objective function for optimization
    def objective_function(pair_indices):
        selected_prices = df.iloc[:, list(pair_indices)]
        profit_loss = calculate_profit_loss(selected_prices, k)
        risk = np.std(profit_loss)
        return np.mean(profit_loss) - theta * risk

    # Define the gradient of the objective function (you need to fill this in)
    def gradient_function(pair_indices):
        # This needs to be calculated or approximated
        return np.array([0, 0])  # Placeholder

    # Projection function to ensure pair indices are within bounds and integers
    def project(pair_indices):
        projected_indices = np.clip(pair_indices, 0, df.shape[1] - 1)
        return np.round(projected_indices).astype(int)

    # Initialization
    num_assets = df.shape[1]
    pair_indices = np.random.choice(num_assets, 2, replace=False)  # Random initial pair

    # Iterative update
    for _ in range(max_iter):
        grad = gradient_function(pair_indices)
        pair_indices = pair_indices.astype(float)
        pair_indices -= learning_rate * grad  # Gradient step
        pair_indices = project(pair_indices)  # Projection step

        # Early stopping condition or convergence check can be added here

    # Final objective value and pair
    optimized_pair = df.columns[pair_indices]
    optimized_result = {
        'pair': optimized_pair,
        'objective_value': objective_function(pair_indices)
    }

    return optimized_result


In [6]:
# Function to parallelize tasks (if needed)
#  def parallelize_task(func, args, num_processes=None):
#      pool = mp.Pool(processes=num_processes)
#      results = pool.starmap(func, args)
#      pool.close()
#      pool.join()
#      return results

In [19]:
def main():
    # Load and preprocess datasets
    wilshire5000 = load_and_preprocess_data('Wilshire5000.csv')
    russell1000 = load_and_preprocess_data('Russell1000.csv')

    # Combine both datasets for a comprehensive pairs analysis
    combined_data = pd.concat([wilshire5000, russell1000], axis=1)
    # combined_data = russell1000
    # Parameters
    theta = 100000  # Example value for theta
    k = 10000       # Example investment amount

    # Prepare a list to store optimization results
    optimization_results = []

    # Iterate over all pairs
    num_assets = combined_data.shape[1]
    for i in range(num_assets):
        for j in range(i + 1, num_assets):
            # Select the pair of assets
            pair_data = combined_data.iloc[:, [i, j]]

            # Optimize for the selected pair
            optimized_result = optimize_pairs(pair_data, theta, k)

            # Append the result
            optimization_results.append(optimized_result)

            # Print progress
            # print(f"Optimized pair {i+1}-{j+1}/{num_assets*(num_assets-1)//2}: {optimized_result}")

        # Convert results to DataFrame
        # Initialize an empty list to store the optimization results
        results = []
        optimized_result = []
        # Find the pair with the best objective value
        best_pair = max(optimization_results, key=lambda x: x['objective_value'])


        # Iterate over each optimization result
        for result in optimization_results:
            # Extract the pair of assets and the objective value
            asset1 = result['pair'][0]
            asset2 = result['pair'][1]
            objective_value = result['objective_value']

            # Create a dictionary with the extracted values
            result_dict = {
                'Asset 1': asset1,
                'Asset 2': asset2,
                'Objective Value': objective_value
            }


            # Append the dictionary to the results list
            if(objective_value < 0):
                results.append(result_dict)



        for result in results:
          print(result)



    # Create a DataFrame from the results list
    # results_df = pd.DataFrame(results)

    # # Save results to CSV file
    # results_df.to_csv('pairs_trading_optimization_results.csv', index=False)
    # print("Optimization results saved to 'pairs_trading_optimization_results_rusell1000.csv'")


In [ ]:
if __name__ == "__main__":
    main()

Streaming output truncated to the last 5000 lines.
{'Asset 1': 'COMM.Close', 'Asset 2': 'ADI.Close', 'Objective Value': -1873.4052366620363}
{'Asset 1': 'COP.Close', 'Asset 2': 'ADI.Close', 'Objective Value': -16550.88104099516}
{'Asset 1': 'ADI.Close', 'Asset 2': 'COR.Close', 'Objective Value': -44032.80573138287}
{'Asset 1': 'CPB.Close', 'Asset 2': 'ADI.Close', 'Objective Value': -14710.746007195858}
{'Asset 1': 'CR.Close', 'Asset 2': 'ADI.Close', 'Objective Value': -2504.6034534237224}
{'Asset 1': 'CRL.Close', 'Asset 2': 'ADI.Close', 'Objective Value': -1798.0220693454758}
{'Asset 1': 'ADI.Close', 'Asset 2': 'CRM.Close', 'Objective Value': -86258.22244193268}
{'Asset 1': 'CSCO.Close', 'Asset 2': 'ADI.Close', 'Objective Value': -13298.475260505313}
{'Asset 1': 'ADI.Close', 'Asset 2': 'CSGP.Close', 'Objective Value': -29831.283857688984}
{'Asset 1': 'ADI.Close', 'Asset 2': 'CSL.Close', 'Objective Value': -1542.2330505368955}
{'Asset 1': 'ADI.Close', 'Asset 2': 'CSX.Close', 'Objective 